In [3]:
!pip install langchain langchain-community faiss-cpu sentence-transformers pypdf groq

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq

# ---------------------------
# GROQ API KEY
# ---------------------------
client = Groq(api_key="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")

# ---------------------------
# Load PDF
# ---------------------------
loader = PyPDFLoader("RAG Based questions.pdf")
documents = loader.load()

# ---------------------------
# Split Text into Chunks
# ---------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

# ---------------------------
# Create Embeddings
# ---------------------------
embeddings = HuggingFaceEmbeddings()

# ---------------------------
# Vector Database
# ---------------------------
vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

# ---------------------------
# Ask Question
# ---------------------------
query = "What is the main topic of the document?"

# Retrieve relevant chunks
retrieved_docs = retriever.invoke(query)

context = "\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}
"""

# ---------------------------
# GROQ LLM CALL
# ---------------------------
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

/tmp/ipykernel_247/566545308.py:32: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The document’s main focus is on how to split (chunk) source material and choose retrieval methods for a Retrieval‑Augmented Generation (RAG) system—covering optimal chunk sizes, the trade‑offs of too‑large or too‑small chunks, and a comparison of keyword‑based versus semantic retrieval.
